28x28x1 -> 5 channels conv 3x3, P = 0, S = 1 <br>
26x26x1 -> 10 channels conv 3x3, P = 0, S = 1 <br>
24x24x1 -> max pool 2x2, P = 0, S = 2 <br>
12x12x1 -> 12 channels conv 3x3, P = 0, S = 1 <br>
10x10x1 -> 15 channels conv 3x3, P = 0, S = 1 <br>
8x8x1 max pool 2x2, P = 0, S = 2 <br>
4x4x1 -> fully connected 40 nodes -> fully connected 10 nodes (for 10 labels) <br>

In [ ]:
import torch
import torch.nn.functional as F
class moderu(torch.nn.Module):
    def __init__(self):
        super(moderu,self).__init__()
        self.conv1 = torch.nn.Conv2d(1,5,3) #input: 1 28x28 channel, kernel 3x3 -> output: 5 features for 5 26x26 channels
        self.conv2 = torch.nn.Conv2d(5,10,3) #input: 5 26x26 channels, kernel 3x3 -> output: 10 features for 10 24x24 channels (maxpooling 2x2 -> 10 12x12 channels)
        self.conv3 = torch.nn.Conv2d(10,12,3) #input: 10 12x12 channels, kernel 3x3 -> output: 15 features for 15 10x10 channels
        self.conv4 = torch.nn.Conv2d(12,15,3) #input: 15 10x10 channels, kernel 3x3 -> output: 5 features for 5 8x8 channels (maxpooling 2x2 -> 5 4x4 channels)
        self.lin1 = torch.nn.Linear(15*4*4,40) #fully connected layer, 40 nodes
        self.lin2 = torch.nn.Linear(40,2) #fully connected layer, 10 nodes (for 10 labels)
    def forward(self,x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(F.relu(self.conv2(x)),2)
        x = F.relu(self.conv3(x))
        x = F.max_pool2d(F.relu(self.conv4(x)),2)
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.lin1(x))
        x = F.softmax(self.lin2(x), dim = 1)
        return x
    def num_flat_features(self,x):
        size = x.size()[1:]  # all dimensions except the batch dimension
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
    print(f"Using device: {torch.cuda.get_device_name(device)}")
else:
    device = "cpu"
    print(f"Using device: CPU")

Using device: Tesla T4


load model

In [ ]:
!pip3 install torchinfo
import torchinfo
from torchinfo import summary
model = moderu()
print(model)
print(summary(model))

moderu(
  (conv1): Conv2d(1, 5, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(5, 10, kernel_size=(3, 3), stride=(1, 1))
  (conv3): Conv2d(10, 12, kernel_size=(3, 3), stride=(1, 1))
  (conv4): Conv2d(12, 15, kernel_size=(3, 3), stride=(1, 1))
  (lin1): Linear(in_features=240, out_features=40, bias=True)
  (lin2): Linear(in_features=40, out_features=2, bias=True)
)
Layer (type:depth-idx)                   Param #
moderu                                   --
├─Conv2d: 1-1                            50
├─Conv2d: 1-2                            460
├─Conv2d: 1-3                            1,092
├─Conv2d: 1-4                            1,635
├─Linear: 1-5                            9,640
├─Linear: 1-6                            82
Total params: 12,959
Trainable params: 12,959
Non-trainable params: 0


load dataset

In [ ]:
# import zipfile
# import torch
# import os
# from torch.utils.data import DataLoader
# from torchvision import datasets, transforms

# with zipfile.ZipFile("dataset_ab.zip", "r") as zip_ref:
#         print("Unzipping data...")
#         zip_ref.extractall("dataset_ab")

# TRAIN_DATAPATH = "dataset_ab/dataset_ab/train"
# TEST_DATAPATH = "dataset_ab/dataset_ab/test"
# VALIDATION_DATAPATH = "dataset_ab/dataset_ab/validation"

# #transform
# transform = transforms.Compose([
#     transforms.Resize(size=(28, 28)),
#     transforms.Grayscale(num_output_channels=1),
#     transforms.ToTensor(),               # Convert images to PyTorch tensors
#     transforms.Normalize((0.5,), (0.5,))  # Normalize the images with mean=0.5 and std=0.5
# ])

# # load dataset
# train_dataset2 = datasets.ImageFolder(root=TRAIN_DATAPATH, transform=transform)
# test_dataset2 = datasets.ImageFolder(root=TEST_DATAPATH, transform=transform)
# validation_dataset2 = datasets.ImageFolder(root=VALIDATION_DATAPATH, transform=transform)

# # # load nmist dataset
# # train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform) #array of tensors
# # test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform) #array of tensors

# # dataloader for batching and shuffling
# train_loader = DataLoader(train_dataset2, batch_size=64, shuffle=True)
# test_loader = DataLoader(test_dataset2, batch_size=64, shuffle=False)
# validation_loader = DataLoader(validation_dataset2, batch_size=64, shuffle=False)

# # https://github.com/notkuroneko/HandWritingRecognition_on_STM32_gathering_data/upload/notkuroneko-patch-1/dataset_NIST


#======================================================================================================================================
# test the file a.txt and b.txt
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader

# File paths
path_a = "/content/a.txt"
path_b = "/content/b.txt"

# Function to load data
def load_data(filename, label):
    with open(filename, 'r') as file:
        lines = file.readlines()

    data = []
    for line in lines:
        stripped_line = line.strip().rstrip(',')  # Remove trailing spaces and commas
        if stripped_line:  # Skip empty lines
            try:
                # Convert line to 28x28 matrix
                matrix = np.array(list(map(int, stripped_line.split(','))), dtype=np.float32).reshape(28, 28)
                data.append(matrix)
            except ValueError as e:
                print(f"Skipping malformed line: {stripped_line} - {e}")

    labels = [label] * len(data)
    return np.array(data), np.array(labels)

# Load the files
data_a, labels_a = load_data(path_a, label=0)
data_b, labels_b = load_data(path_b, label=1)

# Combine and process data
data = np.concatenate((data_a, data_b), axis=0)
labels = np.concatenate((labels_a, labels_b), axis=0)

# Split into training and testing datasets
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)

# Ensure the data has the correct shape: [batch_size, 1, 28, 28]
X_train = X_train[..., np.newaxis]  # Add channel dimension (1, 28, 28)
X_test = X_test[..., np.newaxis]    # Add channel dimension (1, 28, 28)

# Correcting the shape to match [batch_size, channels, height, width]
X_train = np.transpose(X_train, (0, 3, 1, 2))  # Shape should be [batch_size, 1, 28, 28]
X_test = np.transpose(X_test, (0, 3, 1, 2))    # Shape should be [batch_size, 1, 28, 28]

print(f"Data ready! Train size: {len(X_train)}, Test size: {len(X_test)}")

# Prepare PyTorch datasets
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                               torch.tensor(y_train, dtype=torch.long))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                              torch.tensor(y_test, dtype=torch.long))

# Create DataLoaders
batch_size = 32  # Adjust batch size as needed
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("DataLoaders are ready!")



Data ready! Train size: 4037, Test size: 1010
DataLoaders are ready!


In [ ]:
# import random
# from pathlib import Path
# from PIL import Image

# # Set seed
# random.seed(42) # <- try changing this and see what happens
# image_path = Path("dataset_ab/dataset_ab")
# # 1. Get all image paths (* means "any combination")
# image_path_list = list(image_path.glob("*/*/*.png"))

# # 2. Get random image path
# random_image_path = random.choice(image_path_list)

# # 3. Get image class from path name (the image class is the name of the directory where the image is stored)
# image_class = random_image_path.parent.stem

# # 4. Open image
# img = Image.open(random_image_path)

# # 5. Print metadata
# print(f"Random image path: {random_image_path}")
# print(f"Image class: {image_class}")
# print(f"Image height: {img.height}")
# print(f"Image width: {img.width}")
# img

In [ ]:
from tqdm import tqdm

train model

In [ ]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
num_epochs = 50


for epoch in range(num_epochs):

    model.train()
    for inputs, labels in tqdm(train_loader):
        # inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad() #zero the param gradient
        outputs = model(inputs) #feed forward
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        #running_loss += loss.item()
        #_, predicted = torch.max(outputs.data, 1)
        # total += labels.size(0)
        # correct += (predicted == labels).sum().item()
    model.eval()
    with torch.no_grad():
      correct = 0
      total = 0
      running_loss = 0.0
      for inputs, labels in tqdm(test_loader):
          # inputs, labels = inputs.to(device), labels.to(device)
          outputs = model(inputs) #feed forward
          loss = criterion(outputs, labels)
          running_loss += loss.item()
          _, predicted = torch.max(outputs.data, 1)
          total += labels.size(0)
          correct += (predicted == labels).sum().item()
      # Print loss and accuracy statistics after each epoch
      print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(test_loader):.4f}, Accuracy: {100 * correct / total:.2f}%")

100%|██████████| 32/32 [00:00<00:00, 202.12it/s]


Epoch [1/50], Loss: 0.6947, Accuracy: 48.32%


100%|██████████| 32/32 [00:00<00:00, 223.37it/s]


Epoch [2/50], Loss: 0.6944, Accuracy: 48.32%


100%|██████████| 32/32 [00:00<00:00, 217.50it/s]


...

Epoch [48/50], Loss: 0.3487, Accuracy: 96.73%


100%|██████████| 32/32 [00:00<00:00, 215.50it/s]


Epoch [49/50], Loss: 0.3483, Accuracy: 96.73%


100%|██████████| 32/32 [00:00<00:00, 216.58it/s]

Epoch [50/50], Loss: 0.3485, Accuracy: 96.73%


In [ ]:
from google.colab import files
torch.save(model.state_dict(), 'model.pth')
files.download('model.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
trained_moderu = torch.load('model.pth')
print(trained_moderu)

<ipython-input-28-22c422c7be22>:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  trained_moderu = torch.load('model.pth')


OrderedDict([('conv1.weight', tensor([[[[-0.1569, -0.3268, -0.3419],
          [-0.2008,  0.3100,  0.2468],
          [ 0.0792, -0.0406, -0.2571]]],


        [[[-0.2270,  0.1844,  0.0675],
          [-0.2637, -0.2211,  0.3039],
          [ 0.1578,  0.5181,  0.4491]]],


        [[[ 0.1079,  0.0222, -0.2201],
          [ 0.1724, -0.0169, -0.3124],
          [-0.1593, -0.2752,  0.0259]]],


        [[[-0.3173, -0.1234,  0.0851],
          [-0.2873, -0.2325,  0.2292],
          [-0.1259,  0.3434,  0.4819]]],


        [[[-0.0822, -0.2561,  0.2879],
          [-0.2203, -0.2232, -0.2047],
          [-0.0913, -0.0468,  0.1892]]]])), ('conv1.bias', tensor([ 0.3240, -0.0011,  0.2492,  0.5387,  0.5327])), ('conv2.weight', tensor([[[[-1.4241e-01, -9.6108e-02, -8.1245e-02],
          [-1.5298e-01,  1.1534e-01,  1.6128e-01],
          [-1.6367e-02, -7.9673e-02, -6.3597e-02]],

         [[-4.0529e-03, -8.2138e-02,  1.7801e-01],
          [-3.2561e-02,  7.6211e-02,  9.5942e-02],
          [ 2.6231e

In [ ]:
import sys
import numpy as np
np.set_printoptions(threshold=sys.maxsize)
trained_moderu['conv1.weight'].cpu().numpy().tofile('conv1_weight.txt', sep=', ', format='%s')
trained_moderu['conv1.bias'].cpu().numpy().tofile('conv1_bias.txt', sep=', ', format='%s')
trained_moderu['conv2.weight'].cpu().numpy().tofile('conv2_weight.txt', sep=', ', format='%s')
trained_moderu['conv2.bias'].cpu().numpy().tofile('conv2_bias.txt', sep=', ', format='%s')
trained_moderu['conv3.weight'].cpu().numpy().tofile('conv3_weight.txt', sep=', ', format='%s')
trained_moderu['conv3.bias'].cpu().numpy().tofile('conv3_bias.txt', sep=', ', format='%s')
trained_moderu['conv4.weight'].cpu().numpy().tofile('conv4_weight.txt', sep=', ', format='%s')
trained_moderu['conv4.bias'].cpu().numpy().tofile('conv4_bias.txt', sep=', ', format='%s')
trained_moderu['lin1.weight'].cpu().numpy().tofile('lin1_weight.txt', sep=', ', format='%s')
trained_moderu['lin1.bias'].cpu().numpy().tofile('lin1_bias.txt', sep=', ', format='%s')
trained_moderu['lin2.weight'].cpu().numpy().tofile('lin2_weight.txt', sep=', ', format='%s')
trained_moderu['lin2.bias'].cpu().numpy().tofile('lin2_bias.txt', sep=', ', format='%s')
# f = open("result.txt", "w")
# f.write("Data")
# f.close()
# f = open("result.txt", "a")
# f.write(trained_moderu['conv1.bias'].cpu().numpy())
files.download('conv1_weight.txt')
files.download('conv1_bias.txt')
files.download('conv2_weight.txt')
files.download('conv2_bias.txt')
files.download('conv3_weight.txt')
files.download('conv3_bias.txt')
files.download('conv4_weight.txt')
files.download('conv4_bias.txt')
files.download('lin1_weight.txt')
files.download('lin1_bias.txt')
files.download('lin2_weight.txt')
files.download('lin2_bias.txt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>